# Non-linear static analysis with the arc-length methods

The arc-length methods of
[structsolve](https://github.com/saullocastro/structsolve) trace equilibrium
paths that include limit points, where the Newton-Raphson method with load
control fails. The methods of the `Shell` object are passed directly
to `structsolve.Analysis`, selecting the Riks method with
`NL_method='arc_length_riks'`, or the Crisfield method with
`NL_method='arc_length_crisfield'`.

The example below is the snap-through benchmark of Sabir and Lock (1972): a
shallow cylindrical panel with the straight edges hinged and the curved edges
free, loaded by a point force at its center. With a reference load larger
than the limit load, the analysis passes the limit point, where the load
factor decreases during the snap-through, and continues until the load factor
reaches exactly 1.0. The code is extracted from one of the `panels` unit
tests:

In [1]:
import numpy as np
from structsolve import Analysis
from panels.shell import Shell

In [2]:
# from tests/tests_shell/test_nonlinear_riks.py
def test_riks_hinged_cylindrical_shell():
    s = Shell()
    s.model = 'cylshell_clpt_donnell'
    R = 2540.
    theta = 0.2
    s.a = 508. # length of the free curved edges along x
    s.b = R*theta # length of the hinged straight edges along y
    s.r = R
    s.stack = [0.]
    s.plyt = 12.7
    s.laminaprop = (3102.75, 0.3) # isotropic
    s.m = 6
    s.n = 6
    s.nx = 2*s.m
    s.ny = 2*s.n
    for edge in ('x1', 'x2', 'y1', 'y2'):
        for field in 'uvw':
            setattr(s, edge + field, 1.)
            setattr(s, edge + field + 'r', 1.)
    # hinged straight edges
    for edge in ('y1', 'y2'):
        for field in 'uvw':
            setattr(s, edge + field, 0.)

    # reference load larger than the limit load, such that the analysis must
    # pass the limit point and the snap-through before reaching a load factor
    # of 1
    Pref = 4000.
    s.add_point_load(s.a/2., s.b/2., 0., 0., -Pref, cte=False)

    an = Analysis(s.calc_fext, s.calc_fint, s.calc_kC, s.calc_kG)
    an.NL_method = 'arc_length_riks'
    an.initialInc = 0.05
    an.maxArcLength = 50.
    # the assertions below check the path between the converged increments,
    # e.g. the minimum load after the snap-through; capping the increment
    # samples the path finely enough for any convergence criterion, whose
    # iteration counts drive the growth of the arc length
    an.maxInc = 0.02
    increments, cs = an.static(NLgeom=True, silent=True)

    lbds = np.asarray(increments)
    wc = np.array([s.uvw(c=c, gridx=3, gridy=3)[1]['w'][1, 1] for c in cs])
    for lbd, w in zip(lbds, wc):
        print('  load %7.1f N, center deflection %8.4f mm' % (lbd*Pref, w))

    assert np.isclose(lbds[-1], 1.)

    # every converged increment is in equilibrium
    fext = s.calc_fext()
    for lbd, c in zip(lbds, cs):
        fint = s.calc_fint(c=c)
        R = fint - lbd*fext
        assert np.linalg.norm(R) <= 1.e-5*max(np.linalg.norm(lbd*fext),
                                              np.linalg.norm(fint))

    # no snap-back for this thickness: the deflection increases monotonically
    assert np.all(np.diff(wc) < 0)

    # limit point: first converged increment followed by a load drop
    ilim = np.argmax(np.diff(lbds) < 0)
    assert 0 < ilim
    assert np.isclose(lbds[ilim]*Pref, 2280., rtol=0.02)
    # the load is nearly constant around the limit point, such that the
    # deflection at the limit point is checked against the two converged
    # increments that bracket it rather than against the one with the largest
    # load, which depends on how the arc length samples the path
    assert (1 + 0.1)*wc[ilim + 1] <= -8.83 <= (1 - 0.1)*wc[ilim]

    # snap-through: after the limit point the load drops to a fraction of the
    # limit load and then increases again, up to the reference load
    imin = ilim + np.argmin(lbds[ilim:])
    assert ilim < imin < len(lbds) - 1
    assert lbds[imin] < 0.3*lbds[ilim]
    assert np.all(np.diff(lbds[:ilim + 1]) > 0)
    assert np.all(np.diff(lbds[ilim:imin + 1]) < 0)
    assert np.all(np.diff(lbds[imin:]) > 0)
    assert np.isclose(wc[-1], -29.1, rtol=0.02)

In [3]:
test_riks_hinged_cylindrical_shell()
print('test_riks_hinged_cylindrical_shell: passed')

  load   199.4 N, center deflection  -0.4044 mm
  load   397.4 N, center deflection  -0.8280 mm
  load   594.0 N, center deflection  -1.2737 mm
  load   789.0 N, center deflection  -1.7449 mm
  load   982.3 N, center deflection  -2.2459 mm
  load  1173.6 N, center deflection  -2.7827 mm
  load  1362.7 N, center deflection  -3.3637 mm
  load  1549.0 N, center deflection  -4.0008 mm
  load  1732.0 N, center deflection  -4.7132 mm
  load  1910.3 N, center deflection  -5.5348 mm
  load  2081.4 N, center deflection  -6.5368 mm
  load  2235.1 N, center deflection  -7.9372 mm
  load  2234.4 N, center deflection -10.6436 mm
  load  2101.4 N, center deflection -11.7653 mm
  load  1945.9 N, center deflection -12.5818 mm
  load  1790.3 N, center deflection -13.2084 mm
  load  1635.8 N, center deflection -13.7363 mm
  load  1482.6 N, center deflection -14.2095 mm
  load  1330.9 N, center deflection -14.6558 mm
  load  1180.6 N, center deflection -15.0973 mm
  load  1032.0 N, center deflection -15.